In [ ]:
import os
import treecorr
import sys
# 1. 设置 MPI 编译器路径（针对 OpenMPI）
os.environ['OMPI_CC'] = '/opt/homebrew/bin/clang'
os.environ['OMPI_CXX'] = '/opt/homebrew/bin/clang++'
# 2. 配置包含路径（需替换实际 conda 路径）
conda_prefix = os.environ.get('CONDA_PREFIX', '/Users/你的用户名/anaconda3/envs/环境名')
os.environ['C_INCLUDE_PATH'] = f"{conda_prefix}/include:{conda_prefix}/include/python{sys.version_info.major}.{sys.version_info.minor}"
# 3. 添加 Homebrew 的包含路径（M1/M2必备）
os.environ['CPATH'] = f"/opt/homebrew/include:{os.environ.get('CPATH', '')}"
os.environ["OMP_NUM_THREADS"] = "12"
print(treecorr.get_omp_threads())


In [1]:
import os
import treecorr
import sys

In [ ]:
# %matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

def loadfield2d(fn):
    """
    fn: filename
    Returns:
    a: 2D field
    """
    fid = open(fn, 'rb')
    # fid = open(fn, 'rb', 'big')  # big endian
    p1 = np.fromfile(fid, dtype=np.float32)  # real*4
    fid.close()
    n = round(len(p1) ** (1/2))
    if (n**2 != len(p1)):
        print('shape no mach')
        print(n,n**2,len(p1))
        return 0,0
    a = np.reshape(p1, (n, n),order='F')
    # print(fn,n)
    return a,n

def cosmic_extrema(fn, sigma=5.0, exclude_ratio=0.05, visualize=True, 
                  min_depth_ratio=0.12, cluster_radius=6):
    """
    宇宙大尺度结构极值探测器（支持周期性边界条件）
    
    参数:
        delta (np.ndarray): [ng,ng]二维密度场
        sigma (float): 高斯平滑尺度（建议值: 粒子网格尺寸的2-5倍）
        exclude_ratio (float): 边界排除比例 (0.1表示排除10%边界)
        visualize (bool): 是否生成诊断图像

    返回:
        (max_coords, min_coords): 极大/极小值的网格坐标数组
    """

    delta,ng = loadfield2d(fn)
    assert delta.shape == (ng, ng), "必须为方形网格"
    print(delta.max(), delta.min())
    
    # 核心处理流程
    def _periodic_gradient(f):
        """傅里叶周期性导数计算"""
        ky, kx = np.meshgrid(
            2j*np.pi*np.fft.fftfreq(ng),
            2j*np.pi*np.fft.fftfreq(ng),
            indexing='ij'
        )
        fk = np.fft.fft2(f)
        return np.real(np.fft.ifft2(fk * kx)), np.real(np.fft.ifft2(fk * ky))
    
    def apply_boundary_exclusion(mask, edge):
        """统一应用边界排除策略"""
        mask[:edge] = False
        mask[-edge:] = False
        mask[:, :edge] = False
        mask[:, -edge:] = False
        return mask

    # 执行步骤
    smoothed = gaussian_filter(delta, sigma=sigma, mode='wrap')  # 关键：周期平滑
    fx, fy = _periodic_gradient(smoothed)
    
    # 构造Hessian矩阵
    fxx, fxy = _periodic_gradient(fx)
    _, fyy = _periodic_gradient(fy)
    
    # 构建诊断矩阵
    determinant = fxx*fyy - fxy**2
    curvature_condition = determinant > 0  # 非鞍点区域
    
    # 相位图分析 (极大/极小判据)
    maxima_mask = curvature_condition & (fxx < 0) & (fyy < 0)
    minima_mask = curvature_condition & (fxx > 0) & (fyy > 0)
    
    # 边界排除策略
    edge = int(ng * exclude_ratio)
    # 在原有代码中替换为：
    maxima_mask = apply_boundary_exclusion(maxima_mask, edge)
    minima_mask = apply_boundary_exclusion(minima_mask, edge)
    
    # 提取坐标
    max_coords = np.argwhere(maxima_mask)
    min_coords = np.argwhere(minima_mask)


    def filter_extrema(coords, is_maxima=True):
        """三阶段筛选：深度阈值、集群合并、显著性验证"""
        filtered = []
        
        # 阶段1：计算每个极值的深度
        if is_maxima:
            local_avg = np.abs(smoothed[coords[:,0], coords[:,1]] - delta[coords[:,0], coords[:,1]])
            depth = local_avg / delta.max()
            valid = depth > min_depth_ratio
            coords = coords[valid]
            depth = depth[valid]
        else:
            env_radius = int(1.5*sigma)  # 环境计算范围
            neighbors = np.lib.stride_tricks.sliding_window_view(
                delta, (env_radius, env_radius)
            )
            # 计算每个极小值周围区域的均值
            local_avg = np.array([
                neighbors[y, x].mean() 
                for y, x in coords // env_radius
            ])
            depth = -local_avg#np.abs(smoothed[coords[:,0], coords[:,1]]/local_avg)
            print(depth.max(), depth.min())
            valid = depth > min_depth_ratio
            coords = coords[valid]
            depth = -smoothed[coords[:,0], coords[:,1]]/2
            valid = depth > min_depth_ratio
            coords = coords[valid]
            depth = depth[valid]

        # valid = depth > min_depth_ratio
        # coords = coords[valid]
        # depth = depth[valid]
        
        # 阶段2：空间聚类合并
        from sklearn.cluster import DBSCAN
        clustering = DBSCAN(eps=cluster_radius, min_samples=1).fit(coords)
        clusters = np.unique(clustering.labels_)
        
        # 每个簇保留最深点
        for c in clusters:
            mask = clustering.labels_ == c
            cluster_depths = depth[mask]
            deepest_idx = np.argmax(cluster_depths)
            filtered.append(coords[mask][deepest_idx])
        # filtered = coords
            
        return np.array(filtered)

    # # 应用优化筛选
    max_coords = filter_extrema(max_coords, is_maxima=True)
    # min_coords = filter_extrema(min_coords, is_maxima=False)


    # 可视化诊断
    if visualize:

        fig = plt.figure(figsize=(6,6))
        plt.imshow(smoothed, origin='lower')
        plt.clim(-1,3)
        plt.scatter(max_coords[:,1], max_coords[:,0], c='r', s=8, marker='x')
        # plt.scatter(min_coords[:,1], min_coords[:,0], c='b', s=8, marker='o')
        plt.title('pek')
        # fig, ax = plt.subplots(1, 1, figsize=(18,6))
        # plt.set_cmap('viridis')
        
        # ax[0].imshow(delta, origin='lower')
        # ax[0].set_title(f'org σ={sigma}')
        
        # ax[1].imshow(smoothed, origin='lower')
        # ax[1].scatter(max_coords[:,1], max_coords[:,0], c='r', s=8, marker='x')
        # ax[1].scatter(min_coords[:,1], min_coords[:,0], c='b', s=8, marker='o')
        # ax[1].set_title('pek')
        
        # ax[2].imshow(determinant, origin='lower', vmin=-0.1, vmax=0.1)
        # ax[2].set_title('Hessian det')
        
        # plt.tight_layout()
        plt.show()
    
    return max_coords, min_coords

Path = '/mnt/18T/output_2D/3000_3072/3.000_'
maxima, minima = cosmic_extrema(f'{Path}delta_c.bin', sigma=3.0, exclude_ratio=0.05,visualize=True)

print(f"检测到 {len(maxima)} 个极大点 | {len(minima)} 个极小点")

In [3]:
def load2dxpos(fn):
    fid = open(fn, 'rb')
    p1 = np.fromfile(fid, dtype=np.float32)
    fid.close()
    n = round(len(p1)/2)
    if (n*2 != len(p1)):
        print('shape no mach')
        print(n,n**2,len(p1))
        return 0,0
    a = np.reshape(p1, (2, n),order='F')
    # print(fn,n)
    return a,int(np.sqrt(n))
# h_pos,n = load2dxpos(f'{Path}halo_xp_mean_only.bin')

In [ ]:
%matplotlib inline
import numpy as np
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt

def simple_xi(positions, rbins, Lbox):
    """
    计算周期性盒子中的两点相关函数
    
    参数：
    positions : (N,3)数组，粒子三维坐标
    rbins     : 分箱边界的数组（例如 np.logspace）
    Lbox      : 模拟盒子尺寸
    
    返回：
    r_centers : 分箱中心
    xi        : 两点相关函数值
    """
    N = len(positions)
    print(f'计算 {N} 个点的两点相关函数...')
    
    # 创建全连接点对索引 (i,j) 且 i < j
    i, j = np.triu_indices(N, k=1)
    print('创建全连接点对索引...')
    
    # 计算周期性边界下的坐标差
    dx = (positions[i,0] - positions[j,0] + Lbox/2) % Lbox - Lbox/2
    dy = (positions[i,1] - positions[j,1] + Lbox/2) % Lbox - Lbox/2
    print('计算周期性边界下的坐标差...')
    
    # 计算三维距离
    dist = np.sqrt(dx**2 + dy**2)
    print('计算三维距离...')
    
    # 统计距离分箱
    DD_counts, _ = np.histogram(dist, bins=rbins)
    print('统计距离分箱...')
    
    # 计算理论预期值RR
    volumes = np.pi * (rbins[1:]**2 - rbins[:-1]**2)
    RR_counts = 0.5 * N * (N - 1) * (volumes / Lbox**2)
    print('计算理论预期值RR...')
    
    # 计算相关性
    xi = (DD_counts / RR_counts) - 1
    r_centers = (rbins[1:] + rbins[:-1]) / 2
    print('计算相关性...')
    
    return r_centers, xi*r_centers**2


Lbox = 1024
# rbins = np.logspace(-1, 3, 100)
rbins = np.linspace(0, 200, 50)

# # 生成测试数据（1000个随机点+轻微成团性）
# np.random.seed(42)
# N = 10000
# Lbox = 1024
# rand_points = np.random.rand(N, 2) * Lbox
# # r_centers, xi = simple_xi(rand_points, rbins, Lbox)



# # 计算相关性
# r_centers, xi = simple_xi(h_pos.T, rbins, Lbox)
r_centers, xi = simple_xi(maxima[::1], rbins, Lbox)
# r_centers, xi = simple_xi(minima[::1], rbins, Lbox)


# 绘图
plt.figure(figsize=(8,6))
plt.plot(r_centers, xi, 's-', color='darkred', lw=2)
# plt.xscale('log')
# plt.yscale('log')
plt.xlabel('r [Mpc/h]', fontsize=12)
plt.ylabel(r'$r^2\xi(r)$', fontsize=12)
plt.grid(True, which="both", ls="--")
plt.title('2PCF', fontsize=14)
plt.show()  

In [ ]:
import numpy as np
from scipy import fftpack, ndimage
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
import matplotlib.pyplot as plt

# 假设 loadfield2d 函数已定义
# def loadfield2d(filename):
#     # 示例实现
#     data = np.fromfile(filename, dtype=np.float32)
#     ng = int(np.sqrt(data.size)) # 假设是方形网格
#     delta = data.reshape((ng, ng))
#     return delta, ng

def periodic_gaussian_filter(delta, sigma, ng):
    """周期性边界条件下的高斯滤波"""
    # 生成高斯核
    kernel = ndimage.gaussian_filter(np.zeros((ng, ng)), sigma=sigma)
    # 傅里叶变换
    delta_h = fftpack.fft2(delta)
    kernel_h = fftpack.fft2(kernel)
    # 频域乘积
    delta_smoothed_h = delta_h * kernel_h
    # 反变换回空间域
    return fftpack.ifft2(delta_smoothed_h).real

def _periodic_gradient(field):
    """ 带周期边界条件的谱方法梯度计算 """
    ky, kx = np.meshgrid(
        np.fft.fftfreq(field.shape[0]),
        np.fft.fftfreq(field.shape[1]), 
        indexing='ij'
    )
    fk = np.fft.fft2(field)
    dfdx = np.real(np.fft.ifft2(fk * 2j * np.pi * kx))
    dfdy = np.real(np.fft.ifft2(fk * 2j * np.pi * ky))
    return dfdx, dfdy

def hessian_matrix(delta):
    """计算周期性网格上的Hessian矩阵"""
    # 计算一阶导数
    grad_x, grad_y = _periodic_gradient(delta)
    
    # 计算二阶导数
    f_xx, f_xy = _periodic_gradient(grad_x)
    _, f_yy = _periodic_gradient(grad_y)
    
    return grad_x, grad_y, f_xx, f_xy, f_yy

def find_extrema(delta, ng, sigma=2, gradient_threshold=1e-5, epsilon=1e-8):
    """检测极小值候选点"""
    # 高斯滤波平滑
    delta_smoothed = periodic_gaussian_filter(delta, sigma, ng)
    
    # 计算梯度和Hessian矩阵
    gradient_x, gradient_y, f_xx, f_xy, f_yy = hessian_matrix(delta_smoothed)
    
    # 正则化Hessian矩阵（避免数值不稳定）
    f_xx_reg = f_xx + epsilon
    f_yy_reg = f_yy + epsilon
    
    # 计算行列式和迹
    det_hess = f_xx_reg * f_yy_reg - f_xy**2
    trace_hess = f_xx_reg + f_yy_reg
    
    # 筛选极小值候选点：
    # 1. 行列式大于0（特征值同号）
    # 2. 迹大于0（特征值均为正）
    # 3. 梯度足够小（接近临界点）
    min_points = (
        (det_hess > 0) & 
        (trace_hess > 0) & 
        (np.abs(gradient_x) < gradient_threshold) & 
        (np.abs(gradient_y) < gradient_threshold)
    )
    
    # 返回坐标点
    return list(zip(*np.where(min_points)))

def gpr_subpixel_optimization(delta, ng):
    """使用高斯过程回归进行亚像素优化"""
    # 查找极小值候选点
    min_points = find_extrema(delta, ng)
    print(f"找到 {len(min_points)} 个候选点")
    optimized_points = []
    

    # 方案1: 调整RBF核的边界以适应更小的length_scale
    # kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=0.5, length_scale_bounds=(1e-3, 1e2))

    # 方案2: 固定RBF核的length_scale，不让它优化
    # kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds="fixed")

    # 方案3: 使用Matern核(有时对不光滑函数表现更好)，并固定nu=1.5
    from sklearn.gaussian_process.kernels import Matern
    kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=1.5)

    # 对每个极小值候选点进行优化
    for idx, (i, j) in enumerate(min_points):
        # 处理边界情况（使用周期性边界条件）
        i_prev = (i - 1) % ng
        i_next = (i + 1) % ng
        j_prev = (j - 1) % ng
        j_next = (j + 1) % ng
        
        # 提取周围3x3区域（利用周期性边界条件）
        window = delta[np.ix_([i_prev, i, i_next], [j_prev, j, j_next])]
        
        # 构建训练数据 (相对坐标)
        x_coords, y_coords = np.meshgrid([-1, 0, 1], [-1, 0, 1])
        x_train = np.column_stack([x_coords.flatten(), y_coords.flatten()])
        y_train = window.flatten()
        
        # 数据归一化以提高数值稳定性
        y_mean, y_std = y_train.mean(), y_train.std()
        if y_std > 1e-12:  # 防止除零错误
            y_train_normalized = (y_train - y_mean) / y_std
        else:
            y_train_normalized = y_train - y_mean
            
        # 检查窗口是否有足够的变化性，避免病态问题
        if np.max(y_train) - np.min(y_train) < 1e-12:
            # 如果窗口内数值几乎相同，则无需优化，使用原始整数坐标
            optimized_points.append([float(i), float(j)])
            continue
        
        try:
            # 创建GPR模型
            gpr = GaussianProcessRegressor(
                kernel=kernel,
                n_restarts_optimizer=2,  # 进一步减少重启次数以加快速度
                random_state=idx,  # 使用不同的种子
                alpha=1e-7,  # 略微增加对角线加载
                normalize_y=False,  # 我们手动进行了归一化
                optimizer="fmin_l_bfgs_b"
            )
            
            gpr.fit(x_train, y_train_normalized)
            
            # 在局部区域内搜索最小值（适度分辨率网格即可）
            x_test_coords, y_test_coords = np.meshgrid(
                np.linspace(-1, 1, 21), 
                np.linspace(-1, 1, 21)
            )
            x_test = np.column_stack([x_test_coords.flatten(), y_test_coords.flatten()])
            
            # 预测并找出最小值
            y_test_normalized = gpr.predict(x_test, return_std=False)
            min_idx = np.argmin(y_test_normalized)
            
            # 转换回原始坐标系 (亚像素偏移 + 原始整数坐标)
            optimized_point = x_test[min_idx] + [i, j]
            optimized_points.append(optimized_point)
            
        except Exception as e:
            # 如果GPR优化失败，则回退到原始整数坐标
            print(f"警告: 第{idx}个点 GPR优化失败 ({e}), 使用原始坐标")
            optimized_points.append([float(i), float(j)])
        
    return np.array(optimized_points)


Lbox = 3000
Ng = 3072
Redshift = '3.000'

fname = 'E_q'
Path = f'/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/{Lbox:d}_{Ng:d}/{Redshift}_'
fn = f'{Path}{fname}.bin'
delta, ng = loadfield2d(fn)

# 亚像素优化 (内部会调用 find_extrema)
optimized_points = gpr_subpixel_optimization(delta, ng)
print(f"优化后点数: {len(optimized_points)}")

# 可视化
plt.figure(figsize=(8, 8))
plt.imshow(delta, origin='lower', cmap='viridis') # origin='lower' 使 (0,0) 在左下角
plt.colorbar(label='Delta')
# 注意：scatter 的 x 和 y 顺序通常与数组索引相反
plt.scatter(optimized_points[:, 1], optimized_points[:, 0], c='red', s=1, marker='o', alpha=0.5, label='Optimized Minima')
plt.title('Subpixel Optimized Minima')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.clim(-1, 3) # 根据需要调整
plt.show()





In [45]:
np.save('minima_gpr.npy', optimized_points)

In [ ]:
datel,n = loadfield2d(f'{Path}delta_c.bin')
plt.figure(figsize=(4,4),dpi=300)
plt.imshow(datel,cmap='gray',interpolation='none')
plt.clim(-1,3)
plt.colorbar()
plt.axis('equal')
plt.axis('off')
plt.show()

In [ ]:
datel,n = loadfield2d(f'{Path}uD_q.bin')
plt.figure(figsize=(4,4),dpi=300)
plt.imshow(datel.T,cmap='bwr',interpolation='none')
plt.clim(-1,3)
plt.colorbar()
plt.scatter(maxima[:,1]*n/1000, maxima[:,0]*n/1000, c='b', s=0.1, marker='o',alpha=0.1)
plt.axis('equal')
# plt.axis('off')
plt.show()

In [31]:
import numpy as np
from scipy.ndimage import gaussian_filter, uniform_filter
from sklearn.cluster import DBSCAN


def cosmic_extrema(fn=None, delta=None, ng=None, sigma=5.0, exclude_ratio=0.05, 
                   visualize=True, max_depth=0, sigma_decay=0.7,
                   min_depth_ratio=0.08):
    """
    多层宇宙极值探测器 + 亚像素精炼
    
    改进：
    1. 用 Hessian 判别极值
    2. 用 DBSCAN 聚类去掉局部噪声
    3. 用二阶泰勒展开 + 牛顿迭代做亚像素精炼
    """

    def refine_subpixel_newton(dx,dy,dxx,dyy,dxy, coords):
        """局部二阶展开 + Newton refinement"""
        refined = []
        for y, x in coords:
            fx = dx[y, x]
            fy = dy[y, x]
            fxx = dxx[y, x]
            fyy = dyy[y,x]
            fxy = dxy[y,x]
            H = np.array([[fxx, fxy],[fxy, fyy]])
            g = np.array([fx, fy])
            try:
                delta_xy = -np.linalg.solve(H, g)
                if np.all(np.abs(delta_xy) <= 3.0):
                    # 考虑周期性边界条件
                    new_x = (x+delta_xy[1]+float(ng))%float(ng)
                    new_y = (y+delta_xy[0]+float(ng))%float(ng)
                    refined.append([new_y, new_x])
                else:
                    # refined.append([y, x])
                    None
            except np.linalg.LinAlgError:
                # refined.append([y, x])
                None
        return np.array(refined)

    def filter_neighboring(depth, coords,radius):
        filtered = []
        clustering = DBSCAN(eps=radius, min_samples=1).fit(coords)
        clusters = np.unique(clustering.labels_)
        for c in clusters:
            mask = clustering.labels_ == c
            cluster_depths = depth[mask]
            deepest_idx = np.argmax(cluster_depths)
            filtered.append(coords[mask][deepest_idx])
        return np.array(filtered)



    global Ng
    # 初始化
    if delta is None:
        delta, ng = loadfield2d(fn)
        delta0, ng = loadfield2d(fn)
    Ng=ng
    print(delta.max(),delta.min())

    all_min = []
    current_sigma = sigma
    # edge = int(ng * exclude_ratio)

    for depth in range(max_depth+1):
        smoothed = gaussian_filter(delta, sigma=current_sigma, mode='wrap')

        grad_x, grad_y = _periodic_gradient(smoothed)
        # plt.figure(figsize=(12,12))
        # plt.imshow(smoothed,cmap='gray')
        # plt.clim(-1,3)
        # plt.colorbar()
        # plt.show()
        # plt.figure(figsize=(12,12))
        # plt.imshow(grad_x,cmap='bwr')
        # plt.clim(-1,1)
        # plt.colorbar()
        # plt.show()
        # plt.figure(figsize=(12,12))
        # plt.imshow(grad_y,cmap='bwr')
        # plt.clim(-1,1)
        # plt.colorbar()
        # plt.show()
        # return
        hess_xx, hess_xy = _periodic_gradient(grad_x)
        _, hess_yy = _periodic_gradient(grad_y)

        det_hess = hess_xx * hess_yy - hess_xy**2
        if depth == 0:
            maxima = (det_hess > 0) & (hess_xx < 0) & (hess_yy < 0)
        minima = (det_hess > 0) & (hess_xx > 0) & (hess_yy > 0)

        if depth == 0:
            max_coords = np.argwhere(maxima)
            print(f"检测到 {len(max_coords)} 个极大点")
            # local_avg = np.abs(smoothed[max_coords[:,0], max_coords[:,1]] - delta[max_coords[:,0], max_coords[:,1]])
            # depth = local_avg / delta.max()
            # valid = depth > min_depth_ratio
            # max_coords = max_coords[valid]
            # depth = depth[valid]


            max_depth_vals = delta[max_coords[:,0], max_coords[:,1]]
            valid = max_depth_vals > 2
            max_coords = max_coords[valid]
            max_depth_vals = max_depth_vals[valid]
            print(f"检测到 {len(max_coords)} 个极大点")
            max_coords = filter_neighboring(max_depth_vals, max_coords,1)
            print(f"检测到 {len(max_coords)} 个极大点")
            max_coords = refine_subpixel_newton(grad_x,grad_y,hess_xx,hess_yy,hess_xy, max_coords.astype(int))
            print(f"检测到 {len(max_coords)} 个极大点")

        min_coords = np.argwhere(minima)
        print(f"检测到 {len(min_coords)} 个极小点")
        min_depth_vals = delta[min_coords[:,0], min_coords[:,1]]
        min_coords = min_coords[min_depth_vals<-0.5]
        print(f"检测到 {len(min_coords)} 个极小点")
        min_depth_vals = -smoothed[min_coords[:,0], min_coords[:,1]]
        min_coords = filter_neighboring(min_depth_vals, min_coords,3)
        print(f"检测到 {len(min_coords)} 个极小点")
        min_coords = refine_subpixel_newton(grad_x,grad_y,hess_xx,hess_yy,hess_xy, min_coords.astype(int))
        print(f"检测到 {len(min_coords)} 个极小点")
        # min_coords= min_coords[::100,:]
        # plt.figure(figsize=(12,12))
        # plt.imshow(smoothed,cmap='gray')
        # plt.clim(-1,3)
        # plt.scatter(min_coords[:,1], min_coords[:,0], c='r', s=8, marker='x')
        # plt.colorbar()
        # plt.show()
        # return

        all_min.append(min_coords)
        print(f"第 {depth} 层：检测到 {len(min_coords)} 个极小点")

        if visualize:
            plt.figure(figsize=(6,6))
            plt.imshow(delta0)
            plt.colorbar()
            plt.clim(-1,3)
            # plt.scatter(max_coords[:,1], max_coords[:,0], c='r', s=8, marker='x')
            plt.scatter(min_coords[:,1], min_coords[:,0], c='b', s=0.1, marker='o',alpha=0.1)
            plt.title('pek'+str(depth))
            plt.show()

        if depth < max_depth:
            delta = np.zeros_like(delta)
            for (y, x) in min_coords.astype(int):
                delta[y, x] = smoothed[y, x]
            current_sigma /= sigma_decay

    return max_coords*Lbox/ng, min_coords*Lbox/ng
def _periodic_gradient(field):
    """ 带周期边界条件的谱方法梯度计算 """
    ky, kx = np.meshgrid(np.fft.fftfreq(field.shape[0]),
                         np.fft.fftfreq(field.shape[1]), indexing='ij')
    fk = np.fft.fft2(field)
    dfdx = np.real(np.fft.ifft2(fk * 2j * np.pi * kx))
    dfdy = np.real(np.fft.ifft2(fk * 2j * np.pi * ky))
    return dfdx, dfdy

def enhanced_treecorr_xi(positions, rbins, Lbox, show_progress=True):
    """
    高效计算二维周期性盒子两点相关函数
    
    参数增强：
    show_progress : 显示TreeCorr处理进度条
    """
    print(Lbox)
    if (np.max(positions)/Lbox-1 <-1e-2 or np.max(positions) > Lbox) :
        print('positions err')
        return 0,0
    N = positions.shape[0]
    
    # 创建带周期条件的Catalog（改进单位处理）
    data_cat = treecorr.Catalog(
        x=positions[:,0], 
        y=positions[:,1], 
        config={
            'period': Lbox,  # 关键：通过period设置物理尺寸
            'verbose': 2 if show_progress else 0
        }
    )
    
    config = {
        'min_sep': rbins[0] + 1e-5,  # 避免零距离
        'max_sep': rbins[-1],
        'nbins': len(rbins) - 1,
        'bin_type': 'Linear',
        'period': Lbox,
        'bin_slop': 0.04,  # 优化后的精度速度平衡
        'angle_slop': 0.08,
        'metric': 'Periodic',
        'var_method': 'jackknife'  # 可选误差估计方法
    }
    
    # 运行DD计算（增加多线程支持）
    dd = treecorr.NNCorrelation(config)
    dd.process(data_cat, num_threads=12)  # 4线程加速
    
    # 精确理论RR计算（考虑边界条件）
    lower = dd.left_edges
    upper = dd.right_edges
    areas = np.pi * (upper**2 - lower**2)
    density = (N * (N - 1)) / (2 * Lbox**2)  # 点对密度
    rr = areas * density
    
    # 改进的误差处理
    valid = rr > 0
    xi = np.full_like(dd.npairs, -1.0)  # 默认-1
    xi[valid] = (dd.npairs[valid] / rr[valid]) - 1.0
    
    # 添加距离校正因子
    r_weight = dd.meanr**2
    
    return dd.rnom, xi * r_weight

# 可视化增强
def plt_2pcf(positions,rbins,Lbox, figsize=(10,6), ylim=(-0.2,5),name=''):

    r, xi = enhanced_treecorr_xi(positions, rbins, Lbox)
    plt.figure(figsize=figsize)
    plt.errorbar(r, xi, fmt='o-', color='royalblue', 
                markersize=6, linewidth=2, alpha=0.8)
    plt.axhline(0, color='gray', linestyle='--')
    plt.axvline(100, color='red', ls='--')

    plt.xlabel(r'$r\ [\mathrm{Mpc}/h]$', fontsize=12)
        
    plt.ylabel(r'$r^2\xi(r)$', fontsize=12)
    # plt.ylim(ylim)
    plt.title(f'{name} in {fname} 2-Point Correlation Function {Lbox:d}_{Ng:d} ', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:

Lbox = 3000
Ng = 3072
Redshift = '3.000'
fname = 'delta_c'
fname = 'uD_q'
fname = 'E_q'
Path = f'/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/{Lbox:d}_{Ng:d}/{Redshift}_'
maxima, minima = cosmic_extrema(fn=f'{Path}{fname}.bin', sigma=5.0, exclude_ratio=0.05,visualize=False)

# print(f"检测到 {len(maxima)} 个极大点 | {len(minima)} 个极小点")


rbins = np.linspace(0, 200, 40)
# # 计算相关性
# h_pos,n = load2dxpos(f'{Path}halo_xp_mean_only.bin')
# plt_2pcf(h_pos.T*Lbox/Ng, rbins, Lbox,name='Halo')
# plt_2pcf(maxima[::1], rbins, Lbox,name='Maxima')
plt_2pcf(minima[::1], rbins, Lbox,name='Minima')

In [ ]:
#生成高斯分布的Ng*Ng的密度场

fname = 'Gaussian'
delta = np.random.normal(size=(Ng,Ng))


maxima, minima = cosmic_extrema(delta=delta,ng=Ng, sigma=5.0, exclude_ratio=0.05,visualize=False)

# print(f"检测到 {len(maxima)} 个极大点 | {len(minima)} 个极小点")


rbins = np.linspace(0, 200, 40)
# # 计算相关性
# h_pos,n = load2dxpos(f'{Path}halo_xp_mean_only.bin')
# plt_2pcf(h_pos.T*Lbox/Ng, rbins, Lbox,name='Halo')
# plt_2pcf(maxima[::1], rbins, Lbox,name='Maxima')
plt_2pcf(minima[::1], rbins, Lbox,name='Minima')

In [ ]:

fname = 'partical'
v_pos,n = load2dxpos(f'{Path}void_xp_mean_only.bin')
plt_2pcf(v_pos.T*Lbox/Ng, rbins, Lbox,name='Void')

In [ ]:

v_pos,n = load2dxpos(f'{Path}void_qp_mean_only.bin')
plt_2pcf(v_pos.T*Lbox/Ng, rbins, Lbox,name='Halo')

In [ ]:
delta, ng = loadfield2d(fn=f'{Path}{fname}.bin')
h_pos,n = load2dxpos(f'{Path}halo_xp_mean_only.bin')
plt.figure(figsize=(12,12))
plt.imshow(delta,cmap='gray')
plt.clim(-1,3)
# plt.scatter((h_pos.T*Lbox/Ng)[:,1], (h_pos.T*Lbox/Ng)[:,0], c='b', s=8, marker='o')
# plt.scatter(maxima[:,1], maxima[:,0], c='r', s=8, marker='x')
plt.colorbar()
plt.show()

In [ ]:
def load2dvoid(fn):
    with open(fn, 'rb') as f:
        print(fn)
        data = np.fromfile(f, dtype=np.float32)

        n_void = int(len(data)/9)
        print(f"文件中的空洞数量: {n_void}")
        a = data.reshape(9, n_void, order='F')
        print(a.shape)
        center =  a[0:2,:]
        radii =  a[2,:]
        # triangles = a[3:,:]((2,3,n_void))
        triangles = np.zeros((n_void,3, 2))
        triangles[:, 0, :] = a[3:5, :].T  # 点a
        triangles[:, 1, :] = a[5:7, :].T  # 点b
        triangles[:, 2, :] = a[7:9, :].T  # 点c

    return center, radii, triangles

rbins = np.linspace(0, 200, 30)
center, radii, triangles = load2dvoid(f'{Path}DT_void.bin')
h_pos,n = load2dxpos(f'{Path}halo_xp_mean_only.bin')
plt_2pcf(center.T*Lbox/Ng, rbins, Lbox,name='DT_void')
plt_2pcf(h_pos.T*Lbox/Ng, rbins, Lbox,name='Halo')

In [ ]:
import numpy as np
import treecorr
import matplotlib.pyplot as plt

print(treecorr.get_omp_threads())

def enhanced_treecorr_xi(positions, rbins, Lbox, show_progress=True):
    """
    高效计算二维周期性盒子两点相关函数
    
    参数增强：
    show_progress : 显示TreeCorr处理进度条
    """
    N = positions.shape[0]
    
    # 创建带周期条件的Catalog（改进单位处理）
    data_cat = treecorr.Catalog(
        x=positions[:,0], 
        y=positions[:,1], 
        config={
            'period': Lbox,  # 关键：通过period设置物理尺寸
            'verbose': 2 if show_progress else 0
        }
    )
    
    config = {
        'min_sep': rbins[0] + 1e-5,  # 避免零距离
        'max_sep': rbins[-1],
        'nbins': len(rbins) - 1,
        'bin_type': 'Linear',
        'period': Lbox,
        'bin_slop': 0.2,  # 优化后的精度速度平衡
        'metric': 'Periodic',
        'var_method': 'jackknife'  # 可选误差估计方法
    }
    
    # 运行DD计算（增加多线程支持）
    dd = treecorr.NNCorrelation(config)
    dd.process(data_cat, num_threads=12)  # 4线程加速
    
    # 精确理论RR计算（考虑边界条件）
    lower = dd.left_edges
    upper = dd.right_edges
    areas = np.pi * (upper**2 - lower**2)
    density = (N * (N - 1)) / (2 * Lbox**2)  # 点对密度
    rr = areas * density
    
    # 改进的误差处理
    valid = rr > 0
    xi = np.full_like(dd.npairs, -1.0)  # 默认-1
    xi[valid] = (dd.npairs[valid] / rr[valid]) - 1.0
    
    # 添加距离校正因子
    r_weight = dd.meanr**2
    
    return dd.rnom, xi * r_weight

# 可视化增强
def plot_2pcf(positions,rbins,Lbox, figsize=(10,6), ylim=(-0.2,5),name=''):

    r, xi = enhanced_treecorr_xi(positions, rbins, Lbox)
    plt.figure(figsize=figsize)
    plt.errorbar(r, xi, fmt='o-', color='royalblue', 
                markersize=6, linewidth=2, alpha=0.8,label=name)
    plt.axhline(0, color='gray', linestyle='--')
    plt.axvline(100, color='red', ls='--')

    try :
        plt.plot(r_L, r_L**2 * xi_L*10000, label=r'$r^2 \xi(r)$ delta_L')
        plt.plot(r_camb, r_camb**2 * xi_camb*10000, label=r'$r^2 \xi(r)$ CAMB')
        plt.legend()
    except:
        pass
    # if np.any(r > 1000):  # 自动判断是否对数坐标
    #     plt.xscale('log')
    #     plt.xlabel(r'$r\ [\mathrm{Mpc}/h]$', fontsize=12)
    # else:
    plt.xlabel(r'$r\ [\mathrm{Mpc}/h]$', fontsize=12)
        
    plt.ylabel(r'$r^2\xi(r)$', fontsize=12)
    # plt.ylim(ylim)
    plt.title(f'{name} 2-Point Correlation Function', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()




In [ ]:

Path = '/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/3.000_'
pos,n = load2dxpos(f'{Path}xp.bin')
NN = int(1e6)
Lbox = n#3000
# pos = pos/n*Lbox
rbins = np.linspace(0, 200, 30)

# plot_2pcf(pos.T, rbins, Lbox,name='all partical')
# plot_2pcf(pos.T.astype(int).astype(float), rbins, Lbox,name='all partical (int)')

mask = (np.random.rand(NN)*n*n).astype(int)
# print(pos.T[mask,:][0].shape,pos.T.shape)[mask,:]
plot_2pcf(pos.T[:,:], rbins, Lbox,name=f'{NN:d} partical ')
# plot_2pcf(pos.T[:,:], rbins, Lbox,name='all partical ')
# plot_2pcf(pos.T[mask,:].astype(int).astype(float), rbins, Lbox,name='all partical (int)')

In [ ]:
# Luq
# 加载功率谱
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi

import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate, special

# ============ user inputs ============
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/3.000_Cpower_Luq.bin"
# r range to evaluate (Mpc/h if k is in h/Mpc)
r = np.linspace(1.0, 200.0, 400)
# number of points in extended log-k grid (increase for more accuracy)
Nk = 4000
# multipliers to extend original k-range (tune if needed)
kmin_factor = 1e-3
kmax_factor = 1e3
# =====================================

end = 700
k, xi = loadpower(fname)
k = k[1:-end]
pk = xi[1:-end, 5]

# print(k)
# exit

# sort
order = np.argsort(k)
k = k[order]
pk = pk[order]

print("original k range:", k[0], "->", k[-1], "  N=", len(k))

# avoid non-positive Pk
pk = np.where(pk>0, pk, 1e-40)

# fit log-log power laws at ends for smooth extrapolation
m = 8#max(8, len(k)//20)
coef_lo = np.polyfit(np.log(k[:m]), np.log(pk[:m]), 1)
coef_hi = np.polyfit(np.log(k[-m:]), np.log(pk[-m:]), 1)
def pk_low(k): return np.exp(coef_lo[1]) * k**(coef_lo[0])
def pk_high(k): return np.exp(coef_hi[1]) * k**(coef_hi[0])

# extended log-k grid
kmin_ext = k[0] * kmin_factor
kmax_ext = k[-1] * kmax_factor
lnk = np.linspace(np.log(kmin_ext), np.log(kmax_ext), Nk)
k_ln = np.exp(lnk)


print(k.shape,pk.shape)
# interpolate log P inside original range
log_interp = interpolate.interp1d(np.log(k), np.log(pk),
                                  kind='linear', bounds_error=False, fill_value=np.nan)
logpk = log_interp(np.log(k_ln))
pk = np.exp(logpk)

# replace out-of-range by power-law extrapolation
mask_low = k_ln < k[0]
mask_high = k_ln > k[-1]
if np.any(mask_low):
    pk[mask_low] = pk_low(k_ln[mask_low])
if np.any(mask_high):
    pk[mask_high] = pk_high(k_ln[mask_high])

# safety
pk = np.nan_to_num(pk, nan=0.0, posinf=0.0, neginf=0.0)

print("using extended k range:", k_ln[0], "->", k_ln[-1], " Nk=", Nk)

# compute xi(r) via integral in ln k:
# xi(r) = 1/(2π^2) ∫ dk k^2 P(k) j0(kr)
# change var: dk = k d(ln k) -> xi = 1/(2π^2) ∫ d(ln k) k^3 P(k) j0(kr)
def xi_from_pk(r_vals, k, pk, lnk):
    r_vals = np.atleast_1d(r_vals)
    xi = np.zeros_like(r_vals, dtype=float)
    for i, rv in enumerate(r_vals):
        kr = k * rv
        j0 = special.spherical_jn(0, kr)   # sin(kr)/(kr)
        integrand = k**3 * pk * j0
        xi[i] = np.trapz(integrand, lnk) / (2.0 * np.pi**2)
    return xi

xi_u = xi_from_pk(r, k_ln, pk, lnk)

# plt.plot(r, r**2 * xi, label=r'$r^2 \xi(r)$')
# plt.axvline(100, color='red', ls='--')
# plt.xlabel('r [Mpc/h]')
# plt.ylabel(r'$r^2 \xi(r)$')
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

# # print xi at ~100
# idx = np.abs(r-100).argmin()
# print("xi(100) =", xi[idx])

In [ ]:
# LR
# 加载功率谱
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi

import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate, special

# ============ user inputs ============
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/3.000_Cpower_LR.bin"
# r range to evaluate (Mpc/h if k is in h/Mpc)
r = np.linspace(1.0, 200.0, 400)
# number of points in extended log-k grid (increase for more accuracy)
Nk = 4000
# multipliers to extend original k-range (tune if needed)
kmin_factor = 1e-3
kmax_factor = 1e3
# =====================================

end = 700
k, xi = loadpower(fname)
k = k[1:-end]
pk = xi[1:-end, 5]

# print(k)
# exit

# sort
order = np.argsort(k)
k = k[order]
pk = pk[order]

print("original k range:", k[0], "->", k[-1], "  N=", len(k))

# avoid non-positive Pk
pk = np.where(pk>0, pk, 1e-40)

# fit log-log power laws at ends for smooth extrapolation
m = 8#max(8, len(k)//20)
coef_lo = np.polyfit(np.log(k[:m]), np.log(pk[:m]), 1)
coef_hi = np.polyfit(np.log(k[-m:]), np.log(pk[-m:]), 1)
def pk_low(k): return np.exp(coef_lo[1]) * k**(coef_lo[0])
def pk_high(k): return np.exp(coef_hi[1]) * k**(coef_hi[0])

# extended log-k grid
kmin_ext = k[0] * kmin_factor
kmax_ext = k[-1] * kmax_factor
lnk = np.linspace(np.log(kmin_ext), np.log(kmax_ext), Nk)
k_ln = np.exp(lnk)


print(k.shape,pk.shape)
# interpolate log P inside original range
log_interp = interpolate.interp1d(np.log(k), np.log(pk),
                                  kind='linear', bounds_error=False, fill_value=np.nan)
logpk = log_interp(np.log(k_ln))
pk = np.exp(logpk)

# replace out-of-range by power-law extrapolation
mask_low = k_ln < k[0]
mask_high = k_ln > k[-1]
if np.any(mask_low):
    pk[mask_low] = pk_low(k_ln[mask_low])
if np.any(mask_high):
    pk[mask_high] = pk_high(k_ln[mask_high])

# safety
pk = np.nan_to_num(pk, nan=0.0, posinf=0.0, neginf=0.0)

print("using extended k range:", k_ln[0], "->", k_ln[-1], " Nk=", Nk)

# compute xi(r) via integral in ln k:
# xi(r) = 1/(2π^2) ∫ dk k^2 P(k) j0(kr)
# change var: dk = k d(ln k) -> xi = 1/(2π^2) ∫ d(ln k) k^3 P(k) j0(kr)
def xi_from_pk(r_vals, k, pk, lnk):
    r_vals = np.atleast_1d(r_vals)
    xi = np.zeros_like(r_vals, dtype=float)
    for i, rv in enumerate(r_vals):
        kr = k * rv
        j0 = special.spherical_jn(0, kr)   # sin(kr)/(kr)
        integrand = k**3 * pk * j0
        xi[i] = np.trapz(integrand, lnk) / (2.0 * np.pi**2)
    return xi

xi_R = xi_from_pk(r, k_ln, pk, lnk)

# plt.plot(r, r**2 * xi, label=r'$r^2 \xi(r)$')
# plt.axvline(100, color='red', ls='--')
# plt.xlabel('r [Mpc/h]')
# plt.ylabel(r'$r^2 \xi(r)$')
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

# # print xi at ~100
# idx = np.abs(r-100).argmin()
# print("xi(100) =", xi[idx])

In [ ]:
# ic
# 加载功率谱
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi

import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate, special

# ============ user inputs ============
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/200.000_power.bin"
# r range to evaluate (Mpc/h if k is in h/Mpc)
r = np.linspace(1.0, 200.0, 400)
# r=rbins
# r_L=r
# number of points in extended log-k grid (increase for more accuracy)
Nk = 4000
# multipliers to extend original k-range (tune if needed)
kmin_factor = 1e-3
kmax_factor = 1e3
# =====================================

end = 700
k, xi = loadpower(fname)
k = k[1:-end]
pk = xi[1:-end, 2]

# print(k)
# exit

# sort
order = np.argsort(k)
k = k[order]
pk = pk[order]

print("original k range:", k[0], "->", k[-1], "  N=", len(k))

# avoid non-positive Pk
pk = np.where(pk>0, pk, 1e-40)

# fit log-log power laws at ends for smooth extrapolation
m = 8#max(8, len(k)//20)
coef_lo = np.polyfit(np.log(k[:m]), np.log(pk[:m]), 1)
coef_hi = np.polyfit(np.log(k[-m:]), np.log(pk[-m:]), 1)
def pk_low(k): return np.exp(coef_lo[1]) * k**(coef_lo[0])
def pk_high(k): return np.exp(coef_hi[1]) * k**(coef_hi[0])

# extended log-k grid
kmin_ext = k[0] * kmin_factor
kmax_ext = k[-1] * kmax_factor
lnk = np.linspace(np.log(kmin_ext), np.log(kmax_ext), Nk)
k_ln = np.exp(lnk)


print(k.shape,pk.shape)
# interpolate log P inside original range
log_interp = interpolate.interp1d(np.log(k), np.log(pk),
                                  kind='linear', bounds_error=False, fill_value=np.nan)
logpk = log_interp(np.log(k_ln))
pk = np.exp(logpk)

# replace out-of-range by power-law extrapolation
mask_low = k_ln < k[0]
mask_high = k_ln > k[-1]
if np.any(mask_low):
    pk[mask_low] = pk_low(k_ln[mask_low])
if np.any(mask_high):
    pk[mask_high] = pk_high(k_ln[mask_high])

# safety
pk = np.nan_to_num(pk, nan=0.0, posinf=0.0, neginf=0.0)

print("using extended k range:", k_ln[0], "->", k_ln[-1], " Nk=", Nk)

# compute xi(r) via integral in ln k:
# xi(r) = 1/(2π^2) ∫ dk k^2 P(k) j0(kr)
# change var: dk = k d(ln k) -> xi = 1/(2π^2) ∫ d(ln k) k^3 P(k) j0(kr)
def xi_from_pk(r_vals, k, pk, lnk):
    r_vals = np.atleast_1d(r_vals)
    xi = np.zeros_like(r_vals, dtype=float)
    for i, rv in enumerate(r_vals):
        kr = k * rv
        j0 = special.spherical_jn(0, kr)   # sin(kr)/(kr)
        integrand = k**3 * pk * j0
        xi[i] = np.trapz(integrand, lnk) / (2.0 * np.pi**2)
    return xi

xi_L = xi_from_pk(r, k_ln, pk, lnk)

# plt.plot(r, r**2 * xi, label=r'$r^2 \xi(r)$')
# plt.axvline(100, color='red', ls='--')
# plt.xlabel('r [Mpc/h]')
# plt.ylabel(r'$r^2 \xi(r)$')
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

# # print xi at ~100
# idx = np.abs(r-100).argmin()
# print("xi(100) =", xi[idx])

In [ ]:
# 3.000
# 加载功率谱
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi

import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate, special

# ============ user inputs ============
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/3.000_power.bin"
# r range to evaluate (Mpc/h if k is in h/Mpc)
r = np.linspace(1.0, 200.0, 400)
# number of points in extended log-k grid (increase for more accuracy)
Nk = 4000
# multipliers to extend original k-range (tune if needed)
kmin_factor = 1e-3
kmax_factor = 1e3
# =====================================

end = 700
k, xi = loadpower(fname)
k = k[1:-end]
pk = xi[1:-end, 2]

# print(k)
# exit

# sort
order = np.argsort(k)
k = k[order]
pk = pk[order]

print("original k range:", k[0], "->", k[-1], "  N=", len(k))

# avoid non-positive Pk
pk = np.where(pk>0, pk, 1e-40)

# fit log-log power laws at ends for smooth extrapolation
m = 8#max(8, len(k)//20)
coef_lo = np.polyfit(np.log(k[:m]), np.log(pk[:m]), 1)
coef_hi = np.polyfit(np.log(k[-m:]), np.log(pk[-m:]), 1)
def pk_low(k): return np.exp(coef_lo[1]) * k**(coef_lo[0])
def pk_high(k): return np.exp(coef_hi[1]) * k**(coef_hi[0])

# extended log-k grid
kmin_ext = k[0] * kmin_factor
kmax_ext = k[-1] * kmax_factor
lnk = np.linspace(np.log(kmin_ext), np.log(kmax_ext), Nk)
k_ln = np.exp(lnk)


print(k.shape,pk.shape)
# interpolate log P inside original range
log_interp = interpolate.interp1d(np.log(k), np.log(pk),
                                  kind='linear', bounds_error=False, fill_value=np.nan)
logpk = log_interp(np.log(k_ln))
pk = np.exp(logpk)

# replace out-of-range by power-law extrapolation
mask_low = k_ln < k[0]
mask_high = k_ln > k[-1]
if np.any(mask_low):
    pk[mask_low] = pk_low(k_ln[mask_low])
if np.any(mask_high):
    pk[mask_high] = pk_high(k_ln[mask_high])

# safety
pk = np.nan_to_num(pk, nan=0.0, posinf=0.0, neginf=0.0)

print("using extended k range:", k_ln[0], "->", k_ln[-1], " Nk=", Nk)

# compute xi(r) via integral in ln k:
# xi(r) = 1/(2π^2) ∫ dk k^2 P(k) j0(kr)
# change var: dk = k d(ln k) -> xi = 1/(2π^2) ∫ d(ln k) k^3 P(k) j0(kr)
def xi_from_pk(r_vals, k, pk, lnk):
    r_vals = np.atleast_1d(r_vals)
    xi = np.zeros_like(r_vals, dtype=float)
    for i, rv in enumerate(r_vals):
        kr = k * rv
        j0 = special.spherical_jn(0, kr)   # sin(kr)/(kr)
        integrand = k**3 * pk * j0
        xi[i] = np.trapz(integrand, lnk) / (2.0 * np.pi**2)
    return xi

xi_c = xi_from_pk(r, k_ln, pk, lnk)

# plt.plot(r, r**2 * xi, label=r'$r^2 \xi(r)$')
# plt.axvline(100, color='red', ls='--')
# plt.xlabel('r [Mpc/h]')
# plt.ylabel(r'$r^2 \xi(r)$')
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

# # print xi at ~100
# idx = np.abs(r-100).argmin()
# print("xi(100) =", xi[idx])

In [ ]:
# CAMB
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate, special

# ============ user inputs ============
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/neutrinos/IC/Pcb_ic.txt"
# r range to evaluate (Mpc/h if k is in h/Mpc)
r = np.linspace(1.0, 200.0, 400)
# r=rbins
# r_camb=r
# number of points in extended log-k grid (increase for more accuracy)
Nk = 4000
# multipliers to extend original k-range (tune if needed)
kmin_factor = 1e-3
kmax_factor = 1e3
# =====================================

data = np.loadtxt(fname)
k_camb = data[:,0].copy()
pk_camb = data[:,1].copy()

# sort
order = np.argsort(k_camb)
k_camb = k_camb[order]
pk_camb = pk_camb[order]

print("original k range:", k_camb[0], "->", k_camb[-1], "  N=", len(k_camb))

# avoid non-positive Pk
pk_camb = np.where(pk_camb>0, pk_camb, 1e-40)

# fit log-log power laws at ends for smooth extrapolation
m = max(8, len(k_camb)//20)
coef_lo = np.polyfit(np.log(k_camb[:m]), np.log(pk_camb[:m]), 1)
coef_hi = np.polyfit(np.log(k_camb[-m:]), np.log(pk_camb[-m:]), 1)
def pk_low(k): return np.exp(coef_lo[1]) * k**(coef_lo[0])
def pk_high(k): return np.exp(coef_hi[1]) * k**(coef_hi[0])

# extended log-k grid
kmin_ext = k_camb[0] * kmin_factor
kmax_ext = k_camb[-1] * kmax_factor
lnk = np.linspace(np.log(kmin_ext), np.log(kmax_ext), Nk)
k = np.exp(lnk)

# interpolate log P inside original range
log_interp = interpolate.interp1d(np.log(k_camb), np.log(pk_camb),
                                  kind='linear', bounds_error=False, fill_value=np.nan)
logpk = log_interp(np.log(k))
pk = np.exp(logpk)

# replace out-of-range by power-law extrapolation
mask_low = k < k_camb[0]
mask_high = k > k_camb[-1]
if np.any(mask_low):
    pk[mask_low] = pk_low(k[mask_low])
if np.any(mask_high):
    pk[mask_high] = pk_high(k[mask_high])

# safety
pk = np.nan_to_num(pk, nan=0.0, posinf=0.0, neginf=0.0)

print("using extended k range:", k[0], "->", k[-1], " Nk=", Nk)

# compute xi(r) via integral in ln k:
# xi(r) = 1/(2π^2) ∫ dk k^2 P(k) j0(kr)
# change var: dk = k d(ln k) -> xi = 1/(2π^2) ∫ d(ln k) k^3 P(k) j0(kr)
def xi_from_pk(r_vals, k, pk, lnk):
    r_vals = np.atleast_1d(r_vals)
    xi = np.zeros_like(r_vals, dtype=float)
    for i, rv in enumerate(r_vals):
        kr = k * rv
        j0 = special.spherical_jn(0, kr)   # sin(kr)/(kr)
        integrand = k**3 * pk * j0
        xi[i] = np.trapz(integrand, lnk) / (2.0 * np.pi**2)
    return xi

xi_camb = xi_from_pk(r, k, pk, lnk)

# # plot
# plt.figure(figsize=(10,4))
# plt.subplot(1,2,1)
# plt.plot(r, xi, label=r'$\xi(r)$')
# plt.axvline(100, color='red', ls='--', label='~100 Mpc/h')
# plt.xlabel('r [Mpc/h]')
# plt.ylabel(r'$\xi(r)$')
# plt.legend()
# plt.grid(True)

# # plt.subplot(1,2,2)
# plt.plot(r, r**2 * xi, label=r'$r^2 \xi(r)$')
# plt.axvline(100, color='red', ls='--')
# plt.xlabel('r [Mpc/h]')
# plt.ylabel(r'$r^2 \xi(r)$')
# plt.legend()
# plt.grid(True)

# plt.tight_layout()
# plt.show()

# # print xi at ~100
# idx = np.abs(r-100).argmin()
# print("xi(100) =", xi[idx])

In [ ]:
# plt power xi
plt.plot(r, r**2 * xi_camb, label=r'$r^2 \xi(r)$ CAMB')
plt.plot(r, r**2 * xi_L, label=r'$r^2 \xi(r)$ delta_L')
plt.plot(r, r**2 * xi_c/200, label=r'$r^2 \xi(r)/200$ delta_c')
plt.plot(r, r**2 * xi_R/200, label=r'$r^2 \xi(r)/200$ delta_R')
plt.plot(r, r**2 * xi_u/2e10, label=r'$r^2 \xi(r)/2e10 \,\, \mu$ ')
plt.axvline(100, color='red', ls='--')
plt.xlabel('r [Mpc/h]')
plt.ylabel(r'$r^2 \xi(r)$')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:

plt.plot(r, r**2 * xi_camb, label=r'$r^2 \xi(r)$ CAMB')
plt.plot(r, r**2 * xi_L, label=r'$r^2 \xi(r)$ delta_L')
plt.plot(r, r**2 * xi_c/200, label=r'$r^2 \xi(r)/200$ delta_c')
plt.plot(r, r**2 * xi_R/200, label=r'$r^2 \xi(r)/200$ delta_R')

plt.axvline(100, color='red', ls='--')
plt.xlabel('r [Mpc/h]')
plt.ylabel(r'$r^2 \xi(r)$')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============ user inputs ============
# Path = "./"
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/1000_1024/neutrinos/IC/Pcb_ic.txt"

# 需要你设定的宇宙学参数（要和 CAMB 的输入保持一致）
h = 0.67
ombh2 = 0.022
omch2 = 0.12
omb = ombh2 / h**2
omc = omch2 / h**2
Omega_m = (ombh2 + omch2) / h**2
Omega_b = omb
Tcmb = 2.725
# =====================================

# ============ load P(k) ============
data = np.loadtxt(fname)
k = data[:,0]   # CAMB 输出的 k, 单位通常 h/Mpc
pk = data[:,1]

# ============ Eisenstein & Hu no-wiggle ============
def EH_nowiggle(k, Omega_m, Omega_b, h, Tcmb=2.725):
    """
    Eisenstein & Hu 1998 no-wiggle transfer function T_nw(k).
    Returns dimensionless transfer function (not normalized).
    """
    # physical densities
    theta = Tcmb/2.7
    omh2 = Omega_m * h**2
    obh2 = Omega_b * h**2

    # redshift of eq
    z_eq = 2.50e4 * omh2 / theta**4
    k_eq = 0.0746 * omh2 / theta**2  # 1/Mpc

    # baryon suppression
    b1 = 0.313 * omh2**(-0.419) * (1 + 0.607 * omh2**0.674)
    b2 = 0.238 * omh2**0.223
    z_d = 1291 * omh2**0.251 / (1 + 0.659*omh2**0.828) * (1 + b1*obh2**b2)

    # sound horizon
    R_d = 31.5 * obh2 / theta**4 * (1e3/z_d)
    R_eq = 31.5 * obh2 / theta**4 * (1e3/z_eq)
    s = (2/(3*k_eq)) * np.sqrt(6/R_eq) * np.log((np.sqrt(1+R_d)+np.sqrt(R_d+R_eq))/(1+np.sqrt(R_eq)))

    # effective shape parameter
    alpha_gamma = 1 - 0.328*np.log(431*omh2)*obh2/omh2 + 0.38*np.log(22.3*omh2)*(obh2/omh2)**2
    Gamma_eff = Omega_m * h * (alpha_gamma + (1-alpha_gamma)/(1+(0.43*k*s)**4))

    q = k * theta**2 / Gamma_eff

    L0 = np.log(2*np.e + 1.8*q)
    C0 = 14.2 + 731/(1+62.5*q)
    T0 = L0 / (L0 + C0*q**2)
    return T0

# compute transfer function squared ~ shape of P(k)
T_nw = EH_nowiggle(k/h, Omega_m, Omega_b, h, Tcmb)  # 注意: k/h 转为 1/Mpc
# rescale so that P_nw(k) ~ matches large-scale amplitude of CAMB P(k)
# simplest: match at lowest k
scale = pk[0] / (T_nw[0]**2 * k[0])
P_nw = scale * T_nw**2 * k

# ============ plot ratio ============
plt.figure(figsize=(8,5))
plt.plot(k, pk/P_nw - 1, label='(P/P_nw - 1)')
plt.axhline(0, color='k', ls='--', alpha=0.5)
plt.xscale('log')
plt.xlabel(r'$k\ [h/\mathrm{Mpc}]$')
plt.ylabel(r'$P/P_{\rm nw} - 1$')
plt.title('BAO wiggles relative to Eisenstein-Hu no-wiggle')
plt.grid(alpha=0.3, which='both')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy import interpolate

# ============ user inputs ============
# Path = "./"   # 改为你文件所在 folder
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/1000_1024/neutrinos/IC/Pcb_ic.txt"
# =====================================

# load
data = np.loadtxt(fname)
k = data[:,0].copy()
pk = data[:,1].copy()

# sort
order = np.argsort(k)
k = k[order]
pk = pk[order]

# basic plot: log-log P(k)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.loglog(k, pk, label='P(k) raw')
plt.xlabel('k [h/Mpc]')
plt.ylabel('P(k)')
plt.title('Power spectrum (log-log)')
plt.grid(True, which='both', ls=':', alpha=0.5)

# ========== make a smooth "no-wiggle" baseline ==========
# Option A: Savitzky-Golay on log-log (works often)
# choose window length (odd) based on number of points; increase to smooth more
N = len(k)
# window must be odd and <= N. choose a fraction of points, e.g., ~N/25, but at least 5
win = int(max(5, (N//25) | 1))  # ensure odd via bitwise or 1
if win >= N:
    win = N-1 if (N-1)%2==1 else N-2
poly = 3
logk = np.log(k)
logpk = np.log(pk.clip(min=1e-40))  # avoid log(0)

# savgol on logpk vs index (better than on logk because k spacing may be irregular)
logpk_smooth = savgol_filter(logpk, window_length=win, polyorder=poly, mode='interp')
pk_smooth = np.exp(logpk_smooth)

# Option B (alternative, commented): do a spline in log-log, or fit wide-window median filter
# spline = interpolate.UnivariateSpline(logk, logpk, s=1.0)
# pk_smooth = np.exp(spline(logk))

# plot smooth
plt.loglog(k, pk_smooth, lw=2, label='smoothed baseline')
plt.legend()

# ========== residuals to highlight wiggles ==========
rel = (pk - pk_smooth) / pk_smooth

plt.subplot(1,2,2)
# plot residual in linear k-space focusing on BAO scales (e.g., k ~ 0.01 - 0.3 h/Mpc)
plt.plot(k, rel, marker='.', ms=3, lw=0.5)
plt.xscale('log')
plt.xlabel('k [h/Mpc]')
plt.ylabel('(P - P_smooth)/P_smooth')
plt.title('Relative residual (highlights wiggles)')
plt.axhline(0, color='k', ls='--', alpha=0.5)
plt.grid(True, which='both', ls=':', alpha=0.5)
plt.tight_layout()
plt.show()

# ========== zoom into k range typical for BAO ==========
# BAO wiggles typically at k ~ 0.02 - 0.3 h/Mpc (period ~ 0.06-0.1 h/Mpc)
kmin, kmax = 0.01, 0.5
mask = (k >= kmin) & (k <= kmax)
plt.figure(figsize=(7,4))
plt.plot(k[mask], rel[mask], marker='.', ms=4, lw=0.6)
plt.xscale('linear')  # linear x better shows oscillations
plt.xlabel('k [h/Mpc]')
plt.ylabel('(P - P_smooth)/P_smooth')
plt.title(f'Zoomed residuals: {kmin} - {kmax} h/Mpc')
plt.grid(alpha=0.4)
plt.show()

# print some diagnostics
print("k range:", k.min(), "->", k.max(), "  N=", N)
print("savgol window used:", win, "polyorder:", poly)
print("peak-to-peak residual (in zoom range):", np.nanmax(rel[mask]) - np.nanmin(rel[mask]))

In [ ]:
# LR
# 加载功率谱
def loadpower(filename):
    n_row_xi = 10
    fid = open(filename, 'rb')  # 以二进制模式打开文件

    xi = np.fromfile(fid, dtype='float32')  # 读取数据到一维数组
    fid.close()

    xi = np.reshape(xi, (int(len(xi) / n_row_xi), n_row_xi))  # 重新形状为二维数组
    ksim = xi[:, 1]  # 提取第二列作为 ksim

    return ksim, xi

import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate, special

# ============ user inputs ============
fname = "/Users/chenbinghang/Library/CloudStorage/OneDrive-个人/陈冰航/天文/sim_2D/3000_3072/3.000_Cpower_LR.bin"
# r range to evaluate (Mpc/h if k is in h/Mpc)
r = np.linspace(1.0, 200.0, 400)
# number of points in extended log-k grid (increase for more accuracy)
Nk = 4000
# multipliers to extend original k-range (tune if needed)
kmin_factor = 1e-3
kmax_factor = 1e3
# =====================================

end = 700
k, xi = loadpower(fname)
k = k[1:-end]
pk = xi[1:-end, 5]

# print(k)
# exit

# sort
order = np.argsort(k)
k = k[order]
pk = pk[order]

print("original k range:", k[0], "->", k[-1], "  N=", len(k))

# avoid non-positive Pk
pk = np.where(pk>0, pk, 1e-40)

# fit log-log power laws at ends for smooth extrapolation
m = 8#max(8, len(k)//20)
coef_lo = np.polyfit(np.log(k[:m]), np.log(pk[:m]), 1)
coef_hi = np.polyfit(np.log(k[-m:]), np.log(pk[-m:]), 1)
def pk_low(k): return np.exp(coef_lo[1]) * k**(coef_lo[0])
def pk_high(k): return np.exp(coef_hi[1]) * k**(coef_hi[0])

# extended log-k grid
kmin_ext = k[0] * kmin_factor
kmax_ext = k[-1] * kmax_factor
lnk = np.linspace(np.log(kmin_ext), np.log(kmax_ext), Nk)
k_ln = np.exp(lnk)


print(k.shape,pk.shape)
# interpolate log P inside original range
log_interp = interpolate.interp1d(np.log(k), np.log(pk),
                                  kind='linear', bounds_error=False, fill_value=np.nan)
logpk = log_interp(np.log(k_ln))
pk = np.exp(logpk)

# replace out-of-range by power-law extrapolation
mask_low = k_ln < k[0]
mask_high = k_ln > k[-1]
if np.any(mask_low):
    pk[mask_low] = pk_low(k_ln[mask_low])
if np.any(mask_high):
    pk[mask_high] = pk_high(k_ln[mask_high])

# safety
pk = np.nan_to_num(pk, nan=0.0, posinf=0.0, neginf=0.0)

print("using extended k range:", k_ln[0], "->", k_ln[-1], " Nk=", Nk)

def xi_from_pk(r_vals, k, pk, lnk):
    r_vals = np.atleast_1d(r_vals)
    xi = np.zeros_like(r_vals, dtype=float)
    for i, rv in enumerate(r_vals):
        kr = k * rv
        j0 = special.spherical_jn(0, kr)   # sin(kr)/(kr)
        integrand = k**3 * pk * j0
        xi[i] = np.trapz(integrand, lnk) / (2.0 * np.pi**2)
    return xi

pi = 1
xi = xi_from_pk(r, k_ln, pk, lnk)
simga = 1
xi_1 = xi_from_pk(r, k_ln, pk * np.exp(-pi*k_ln**2 * simga**2), lnk)
simga = 3
xi_3 = xi_from_pk(r, k_ln, pk * np.exp(-pi*k_ln**2 * simga**2), lnk)
simga = 5
xi_5 = xi_from_pk(r, k_ln, pk * np.exp(-pi*k_ln**2 * simga**2), lnk)

plt.plot(r, r**2 * xi, label=r'$r^2 \xi(r)$')
plt.plot(r, r**2 * xi_1, label=r'$r^2 \xi(r) \exp(-1^2 k^2 )$')
plt.plot(r, r**2 * xi_3, label=r'$r^2 \xi(r) \exp(-3^2 k^2 )$')
plt.plot(r, r**2 * xi_5, label=r'$r^2 \xi(r) \exp(-5^2 k^2 )$')
plt.axvline(100, color='red', ls='--')
plt.xlabel('r [Mpc/h]')
plt.ylabel(r'$r^2 \xi(r)$')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
